# TrafficFlowBench — Official baseline → submission.csv

Runs organizer historical-mean (Task 1), persistence (Task 2), and NNLS ODME (Task 4) for **validation + private**, then merges.

**Expect ~0.55 public** (naive baseline). Use as a smoke-test submit, then improve.

Attach competition data. Optional: turn Accelerator off (CPU is fine).

In [ ]:
import os, sys, subprocess, glob
from pathlib import Path
import pandas as pd

COMP = Path("/kaggle/input/2026-ieee-big-data-traffic-flow-bench")
candidates = [
    COMP / "kaggle_public",
    COMP,
]
REL = None
for c in candidates:
    if (c / "submission_key.csv").exists() and (c / "corridors").exists():
        REL = c
        break
if REL is None:
    # fallback: search one level deep
    hits = list(COMP.rglob("submission_key.csv"))
    if not hits:
        raise FileNotFoundError("Could not find kaggle_public / submission_key.csv under /kaggle/input")
    REL = hits[0].parent

print("REL =", REL)
print("key exists:", (REL / "submission_key.csv").exists())
WORK = Path("/kaggle/working")
PARTS = WORK / "_parts"
PARTS.mkdir(exist_ok=True)

In [ ]:
# Clone official baselines once
REPO = WORK / "trafficflowbench-public"
if not REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/jacky850/trafficflowbench-public.git", str(REPO)
    ], check=True)
SRC = REPO / "src"
sys.path.insert(0, str(SRC))
!pip -q install pyarrow scipy
print("repo ready")

In [ ]:
def run(cmd):
    print("+", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True, cwd=str(REPO))

def concat(parts, out):
    frames = [pd.read_csv(p) for p in parts]
    pd.concat(frames, ignore_index=True).to_csv(out, index=False)
    print(out, sum(len(f) for f in frames))

state_parts, queue_parts, odme_parts = [], [], []
py = sys.executable

for split in ("validation", "private"):
    state_out = PARTS / f"state_{split}.csv"
    queue_out = PARTS / f"queue_{split}.csv"
    odme_dir = PARTS / f"odme_{split}"
    odme_dir.mkdir(exist_ok=True)

    run([py, str(SRC/"task1"/"build_task1_baseline_submission.py"),
         "--release-root", str(REL), "--split", split, "--output", str(state_out)])
    state_parts.append(state_out)

    run([py, str(SRC/"task2"/"build_task2_persistence_submission.py"),
         "--release-root", str(REL), "--split", split, "--output", str(queue_out)])
    queue_parts.append(queue_out)

    run([py, str(SRC/"task4"/"build_task4_odme_artifacts.py"),
         "--release-root", str(REL), "--split", split, "--output-root", str(odme_dir)])
    cand = odme_dir / "baseline_submission.csv"
    if not cand.exists():
        found = list(odme_dir.rglob("baseline_submission.csv"))
        assert found, odme_dir
        cand = found[0]
    odme_parts.append(cand)

state = PARTS / "state_all.csv"
queue = PARTS / "queue_all.csv"
odme = PARTS / "odme_all.csv"
concat(state_parts, state)
concat(queue_parts, queue)
concat(odme_parts, odme)

In [ ]:
out = WORK / "submission.csv"
run([py, str(SRC/"merge_submissions.py"),
     "--state", str(state),
     "--queue", str(queue),
     "--odme", str(odme),
     "--key", str(REL / "submission_key.csv"),
     "--output", str(out)])

sub = pd.read_csv(out, nrows=5)
print(sub)
print("rows:", sum(1 for _ in open(out)) - 1)
print("READY:", out)
print("Use Submit → submit this notebook output, or download submission.csv")